# 🧠 Day 3-5: Fine-Tune BERT for Travel Intent Classification
## Intent-Based Unified Travel Search Using NLP
### By Goureesankar & Vihaan

**What this notebook does:**
1. Loads the dataset you created in Day 1-2 from Google Drive
2. Fine-tunes BERT-base on your travel intent dataset
3. Evaluates on the test set with full metrics
4. Generates paper-ready tables and charts (Table V, Table VI numbers)

**⚠️ IMPORTANT: Enable GPU before running!**
1. Go to **Runtime → Change runtime type**
2. Select **T4 GPU** under Hardware accelerator
3. Click **Save**

Training takes ~30-40 minutes on Colab free tier T4 GPU.

---

## Step 0: Check GPU & Mount Drive

In [ ]:
# Check if GPU is available
import torch
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    print(f'✅ GPU available: {gpu_name}')
    print(f'   GPU Memory: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB')
else:
    print('❌ NO GPU detected!')
    print('   Go to Runtime → Change runtime type → Select T4 GPU')
    print('   Then restart and run this cell again')

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = '/content/drive/MyDrive/NLP_Travel_Search'
MODEL_DIR = f'{PROJECT_DIR}/model'
RESULTS_DIR = f'{PROJECT_DIR}/results'
os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

# Verify dataset exists
assert os.path.exists(f'{PROJECT_DIR}/data/train.csv'), '❌ train.csv not found! Run Day 1-2 notebook first.'
print(f'✅ Project folder found: {PROJECT_DIR}')
print(f'✅ Dataset files found!')

## Step 1: Install Libraries

In [ ]:
!pip install transformers datasets accelerate scikit-learn pandas matplotlib seaborn -q
print('✅ All libraries installed!')

## Step 2: Load Dataset

In [ ]:
import pandas as pd
import json

# Load the splits you created in Day 1-2
train_df = pd.read_csv(f'{PROJECT_DIR}/data/train.csv')
val_df = pd.read_csv(f'{PROJECT_DIR}/data/val.csv')
test_df = pd.read_csv(f'{PROJECT_DIR}/data/test.csv')

# Load label mapping
with open(f'{PROJECT_DIR}/data/label_map.json', 'r') as f:
    INTENT_LABELS = json.load(f)
LABEL_TO_INTENT = {v: k for k, v in INTENT_LABELS.items()}
NUM_LABELS = len(INTENT_LABELS)

print(f'📊 Dataset loaded:')
print(f'   Train: {len(train_df)} samples')
print(f'   Val:   {len(val_df)} samples')
print(f'   Test:  {len(test_df)} samples')
print(f'   Labels: {INTENT_LABELS}')

# Show a few examples
print(f'\n📝 Sample training data:')
for _, row in train_df.sample(5, random_state=42).iterrows():
    print(f'   [{row["intent"]}] "{row["text"]}"')

## Step 3: Tokenize with BERT Tokenizer

In [ ]:
from transformers import BertTokenizer
from torch.utils.data import Dataset, DataLoader

# Load BERT tokenizer
print('Loading BERT tokenizer...')
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
print('✅ Tokenizer loaded!')

# Show how tokenization works
example = "Book a flight to Paris on March 15"
tokens = tokenizer(example, return_tensors='pt')
print(f'\n📝 Example tokenization:')
print(f'   Input: "{example}"')
print(f'   Tokens: {tokenizer.convert_ids_to_tokens(tokens["input_ids"][0])}')
print(f'   Token IDs shape: {tokens["input_ids"].shape}')

In [ ]:
# Create a PyTorch Dataset class
# This is how we feed data to BERT during training

class TravelIntentDataset(Dataset):
    """
    Custom dataset that tokenizes travel queries and returns
    them in the format BERT expects.
    """
    def __init__(self, dataframe, tokenizer, max_length=64):
        self.texts = dataframe['text'].tolist()
        self.labels = dataframe['label'].tolist()
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]

        # Tokenize the text
        encoding = self.tokenizer(
            text,
            max_length=self.max_length,
            padding='max_length',      # Pad short texts to max_length
            truncation=True,            # Cut long texts to max_length
            return_tensors='pt'         # Return PyTorch tensors
        )

        return {
            'input_ids': encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'labels': torch.tensor(label, dtype=torch.long)
        }

# Create datasets
MAX_LENGTH = 64  # Travel queries are short, 64 tokens is enough

train_dataset = TravelIntentDataset(train_df, tokenizer, MAX_LENGTH)
val_dataset = TravelIntentDataset(val_df, tokenizer, MAX_LENGTH)
test_dataset = TravelIntentDataset(test_df, tokenizer, MAX_LENGTH)

print(f'✅ Datasets created!')
print(f'   Train: {len(train_dataset)} samples')
print(f'   Val:   {len(val_dataset)} samples')
print(f'   Test:  {len(test_dataset)} samples')

# Peek at one sample
sample = train_dataset[0]
print(f'\n📝 Sample tensor shapes:')
print(f'   input_ids: {sample["input_ids"].shape}')
print(f'   attention_mask: {sample["attention_mask"].shape}')
print(f'   label: {sample["labels"]}')

## Step 4: Load BERT Model for Classification

In [ ]:
from transformers import BertForSequenceClassification

# Load BERT with a classification head on top
# This adds a linear layer on top of BERT that outputs 5 classes
print('Loading BERT model...')
model = BertForSequenceClassification.from_pretrained(
    'bert-base-uncased',
    num_labels=NUM_LABELS,  # 5 intent categories
    problem_type='single_label_classification'
)

# Move to GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f'✅ BERT model loaded on {device}!')
print(f'   Total parameters: {total_params:,}')
print(f'   Trainable parameters: {trainable_params:,}')
print(f'   Output classes: {NUM_LABELS}')

## Step 5: Set Up Training

We use HuggingFace's `Trainer` class which handles:
- Training loop
- Gradient computation
- Learning rate scheduling
- Evaluation during training
- Saving best model

In [ ]:
from transformers import TrainingArguments, Trainer
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import numpy as np

def compute_metrics(pred):
    """
    This function is called by the Trainer after each evaluation.
    It computes accuracy, precision, recall, and F1.
    """
    labels = pred.label_ids
    preds = np.argmax(pred.predictions, axis=1)

    accuracy = accuracy_score(labels, preds)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average='macro'
    )

    return {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1
    }

# Training configuration
# These are the hyperparameters mentioned in the paper:
# "learning rate 2e-5, batch size 32, 4 epochs with early stopping"

training_args = TrainingArguments(
    output_dir=f'{PROJECT_DIR}/checkpoints',
    
    # Training hyperparameters (THESE GO IN YOUR PAPER)
    num_train_epochs=4,
    learning_rate=2e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    weight_decay=0.01,
    warmup_steps=100,
    
    # Evaluation strategy
    eval_strategy='steps',
    eval_steps=100,              # Evaluate every 100 steps
    save_strategy='steps',
    save_steps=100,
    load_best_model_at_end=True,  # Load best model when training ends
    metric_for_best_model='f1',   # Use F1 to pick best model
    greater_is_better=True,
    save_total_limit=2,           # Keep only 2 best checkpoints (save space)
    
    # Logging
    logging_dir=f'{PROJECT_DIR}/logs',
    logging_steps=50,
    report_to='none',             # Don't send logs to wandb etc.
    
    # Performance
    fp16=True,                    # Use mixed precision (faster on T4 GPU)
    dataloader_num_workers=2,
)

# Create the Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)

print('✅ Training configured!')
print(f'   Epochs: 4')
print(f'   Learning rate: 2e-5')
print(f'   Batch size: 32')
print(f'   Evaluation every 100 steps')
print(f'   Best model selected by F1 score')
print(f'\n⏰ Ready to train! This will take ~30-40 minutes on T4 GPU.')

## Step 6: Train! 🚀

This is the main training cell. It will take **~30-40 minutes** on Colab's free T4 GPU.

You'll see a progress bar and metrics updating as it trains.

In [ ]:
# 🚀 START TRAINING!
print('='*60)
print('  TRAINING BERT FOR TRAVEL INTENT CLASSIFICATION')
print('='*60)
print()

train_result = trainer.train()

print('\n' + '='*60)
print('  ✅ TRAINING COMPLETE!')
print('='*60)
print(f'\n  Training loss: {train_result.training_loss:.4f}')
print(f'  Total steps: {train_result.global_step}')
print(f'  Training time: {train_result.metrics["train_runtime"]:.0f} seconds')

## Step 7: Evaluate on Test Set

Now we evaluate the trained model on the **test set** (data it has never seen).
These are the numbers that go directly into your paper!

In [ ]:
from sklearn.metrics import (
    classification_report, confusion_matrix, accuracy_score,
    precision_recall_fscore_support
)

# Get predictions on test set
print('Running evaluation on test set...')
predictions = trainer.predict(test_dataset)

# Extract predictions and true labels
pred_labels = np.argmax(predictions.predictions, axis=1)
true_labels = predictions.label_ids

# Overall accuracy
accuracy = accuracy_score(true_labels, pred_labels)

# Per-class metrics
target_names = [LABEL_TO_INTENT[i] for i in range(NUM_LABELS)]
report = classification_report(
    true_labels, pred_labels,
    target_names=target_names,
    digits=4,
    output_dict=True
)

# Print the full report
print('\n' + '='*60)
print('  🏆 TEST SET RESULTS (FOR YOUR PAPER)')
print('='*60)
print(f'\n  Overall Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)')
print(f'\n  Per-class results:')
print(classification_report(true_labels, pred_labels, target_names=target_names, digits=4))

## Step 8: Generate Paper-Ready Table V

This creates the exact table you need for your paper:

**TABLE V: Intent Classification Results**

In [ ]:
# ============================================================
# TABLE V FOR YOUR PAPER: Intent Classification Results
# ============================================================

print('\n' + '='*70)
print('  TABLE V: Intent Classification Results')
print('  (Copy these numbers directly into your paper!)')
print('='*70)
print(f'{"Intent Category":<20} {"Precision":<12} {"Recall":<12} {"F1-Score":<12} {"Support":<10}')
print('-'*70)

for intent_name in target_names:
    p = report[intent_name]['precision']
    r = report[intent_name]['recall']
    f = report[intent_name]['f1-score']
    s = int(report[intent_name]['support'])
    print(f'{intent_name:<20} {p:<12.4f} {r:<12.4f} {f:<12.4f} {s:<10}')

print('-'*70)

# Macro average
macro_p = report['macro avg']['precision']
macro_r = report['macro avg']['recall']
macro_f = report['macro avg']['f1-score']
print(f'{"Macro Average":<20} {macro_p:<12.4f} {macro_r:<12.4f} {macro_f:<12.4f}')
print(f'{"Overall Accuracy":<20} {"-":<12} {"-":<12} {accuracy:<12.4f}')
print('='*70)

# Save for later use
paper_results = {
    'overall_accuracy': round(accuracy, 4),
    'macro_precision': round(macro_p, 4),
    'macro_recall': round(macro_r, 4),
    'macro_f1': round(macro_f, 4),
    'per_class': {}
}
for intent_name in target_names:
    paper_results['per_class'][intent_name] = {
        'precision': round(report[intent_name]['precision'], 4),
        'recall': round(report[intent_name]['recall'], 4),
        'f1': round(report[intent_name]['f1-score'], 4),
        'support': int(report[intent_name]['support'])
    }

## Step 9: Confusion Matrix (for the paper)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Compute confusion matrix
cm = confusion_matrix(true_labels, pred_labels)

# Plot it beautifully
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=target_names,
    yticklabels=target_names,
    ax=ax,
    annot_kws={'size': 12}
)
ax.set_xlabel('Predicted Label', fontsize=12, fontweight='bold')
ax.set_ylabel('True Label', fontsize=12, fontweight='bold')
ax.set_title('Confusion Matrix — BERT Travel Intent Classification', fontsize=14, fontweight='bold')
plt.tight_layout()

# Save for paper
plt.savefig(f'{RESULTS_DIR}/confusion_matrix.png', dpi=300, bbox_inches='tight')
plt.show()
print(f'✅ Confusion matrix saved to {RESULTS_DIR}/confusion_matrix.png')
print('   (Use this as a figure in your paper!)')

## Step 10: Training Loss Curve (for the paper)

In [ ]:
# Plot training metrics over time
log_history = trainer.state.log_history

# Extract training loss
train_steps = [x['step'] for x in log_history if 'loss' in x]
train_losses = [x['loss'] for x in log_history if 'loss' in x]

# Extract eval metrics
eval_steps = [x['step'] for x in log_history if 'eval_loss' in x]
eval_losses = [x['eval_loss'] for x in log_history if 'eval_loss' in x]
eval_f1s = [x['eval_f1'] for x in log_history if 'eval_f1' in x]
eval_accs = [x['eval_accuracy'] for x in log_history if 'eval_accuracy' in x]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Plot 1: Training & Validation Loss
axes[0].plot(train_steps, train_losses, 'b-', alpha=0.7, label='Training Loss')
axes[0].plot(eval_steps, eval_losses, 'r-o', markersize=4, label='Validation Loss')
axes[0].set_xlabel('Training Steps')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training & Validation Loss', fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Plot 2: Validation F1 Score
axes[1].plot(eval_steps, eval_f1s, 'g-o', markersize=4, color='#2ca02c')
axes[1].set_xlabel('Training Steps')
axes[1].set_ylabel('Macro F1 Score')
axes[1].set_title('Validation F1 Score', fontweight='bold')
axes[1].grid(True, alpha=0.3)
axes[1].set_ylim([min(eval_f1s) - 0.02, 1.0])

# Plot 3: Validation Accuracy
axes[2].plot(eval_steps, eval_accs, 'm-o', markersize=4, color='#9467bd')
axes[2].set_xlabel('Training Steps')
axes[2].set_ylabel('Accuracy')
axes[2].set_title('Validation Accuracy', fontweight='bold')
axes[2].grid(True, alpha=0.3)
axes[2].set_ylim([min(eval_accs) - 0.02, 1.0])

plt.suptitle('BERT Fine-Tuning Progress for Travel Intent Classification', 
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/training_curves.png', dpi=300, bbox_inches='tight')
plt.show()
print(f'✅ Training curves saved to {RESULTS_DIR}/training_curves.png')

## Step 11: Per-Class F1 Bar Chart (for the paper)

In [ ]:
# Bar chart of per-class F1 scores
fig, ax = plt.subplots(figsize=(10, 6))

f1_scores = [report[name]['f1-score'] for name in target_names]
colors = ['#4472C4', '#ED7D31', '#A5A5A5', '#FFC000', '#5B9BD5']

bars = ax.bar(target_names, f1_scores, color=colors, edgecolor='black', linewidth=0.5)

# Add value labels on bars
for bar, f1 in zip(bars, f1_scores):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.005,
            f'{f1:.4f}', ha='center', va='bottom', fontweight='bold', fontsize=11)

# Add macro average line
ax.axhline(y=macro_f, color='red', linestyle='--', linewidth=1.5, 
           label=f'Macro Avg F1: {macro_f:.4f}')

ax.set_ylabel('F1-Score', fontsize=12, fontweight='bold')
ax.set_xlabel('Intent Category', fontsize=12, fontweight='bold')
ax.set_title('Per-Class F1 Scores — BERT Travel Intent Classification', 
             fontsize=14, fontweight='bold')
ax.set_ylim([min(f1_scores) - 0.05, 1.02])
ax.legend(fontsize=11)
ax.grid(True, axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/f1_scores_bar_chart.png', dpi=300, bbox_inches='tight')
plt.show()
print(f'✅ F1 bar chart saved!')

## Step 12: Save the Trained Model

In [ ]:
# Save the fine-tuned model to Google Drive
# You'll need this for Week 2 (building the full system)

print('Saving model to Google Drive...')
trainer.save_model(f'{MODEL_DIR}/bert-travel-intent')
tokenizer.save_pretrained(f'{MODEL_DIR}/bert-travel-intent')

# Save results for paper
with open(f'{RESULTS_DIR}/classification_results.json', 'w') as f:
    json.dump(paper_results, f, indent=2)

print(f'✅ Model saved to {MODEL_DIR}/bert-travel-intent')
print(f'✅ Results saved to {RESULTS_DIR}/classification_results.json')

# Show what's saved
model_size = sum(
    os.path.getsize(os.path.join(f'{MODEL_DIR}/bert-travel-intent', f))
    for f in os.listdir(f'{MODEL_DIR}/bert-travel-intent')
) / (1024*1024)
print(f'   Model size: {model_size:.0f} MB')

## Step 13: Test with Your Own Queries! 🎉

In [ ]:
def predict_intent(text):
    """
    Predict the travel intent of a query using our fine-tuned BERT model.
    """
    model.eval()
    inputs = tokenizer(
        text, return_tensors='pt', max_length=64,
        padding='max_length', truncation=True
    ).to(device)

    with torch.no_grad():
        outputs = model(**inputs)
        probs = torch.softmax(outputs.logits, dim=1)
        pred_label = torch.argmax(probs, dim=1).item()
        confidence = probs[0][pred_label].item()

    return LABEL_TO_INTENT[pred_label], confidence

# Test with various queries
test_queries = [
    "Book a flight to Tokyo next Friday",
    "What's the weather like in Bali in December?",
    "How do I get from the airport to downtown Paris?",
    "Suggest romantic beach destinations under $2000",
    "Compare Marriott vs Hilton in New York",
    "I need a cheap hostel in Barcelona for next week",
    "Is Goa safe for solo female travelers?",
    "Navigate from Chennai airport to Mahabalipuram",
    "Find me a family-friendly resort with kids activities",
    "Which is better for honeymoon: Maldives or Bali?",
]

print('\n' + '='*70)
print('  🧪 LIVE PREDICTIONS WITH YOUR TRAINED MODEL')
print('='*70)
for query in test_queries:
    intent, conf = predict_intent(query)
    print(f'\n  💬 "{query}"')
    print(f'  → Intent: {intent} (confidence: {conf:.2%})')

In [ ]:
# TRY YOUR OWN QUERIES HERE!
# Change the text below and run this cell

my_query = "Plan a 3-day trip to Kerala with houseboat and Ayurveda spa"
intent, conf = predict_intent(my_query)
print(f'💬 "{my_query}"')
print(f'→ Intent: {intent} (confidence: {conf:.2%})')

## Step 14: Final Summary — Numbers for Your Paper

In [ ]:
print('\n' + '='*70)
print('  📝 FINAL SUMMARY: COPY THESE INTO YOUR PAPER')
print('='*70)

print(f'''
📊 FOR SECTION V.A (Experimental Setup):
  "The intent classification model was fine-tuned from bert-base-uncased
   using HuggingFace Transformers on a dataset combining the ATIS corpus
   ({len(train_df[train_df["source"].str.startswith("atis")])} ATIS samples) and a custom travel
   intent dataset of {len(train_df[train_df["source"]=="custom"])} queries labeled across five
   categories. Training used learning rate 2e-5, batch size 32, 4 epochs
   with early stopping on validation F1. Max sequence length: 64 tokens."

🏆 FOR SECTION V.B (Intent Classification Results):
  "The fine-tuned BERT model achieves an overall accuracy of
   {accuracy*100:.1f}% and macro-averaged F1-score of {macro_f:.3f}
   on the test set."
''')

print(f'\n📊 TABLE V values:')
print(f'{"Intent":<15} {"Precision":<10} {"Recall":<10} {"F1":<10}')
print('-'*45)
for name in target_names:
    p = report[name]['precision']
    r = report[name]['recall']
    f = report[name]['f1-score']
    print(f'{name:<15} {p:.3f}      {r:.3f}      {f:.3f}')
print('-'*45)
print(f'{"Macro Avg":<15} {macro_p:.3f}      {macro_r:.3f}      {macro_f:.3f}')
print(f'{"Accuracy":<15} {"":<10} {"":<10} {accuracy:.3f}')

print(f'''
📁 FILES SAVED TO GOOGLE DRIVE:
  {MODEL_DIR}/bert-travel-intent/    → Trained model (for Week 2)
  {RESULTS_DIR}/confusion_matrix.png       → Figure for paper
  {RESULTS_DIR}/training_curves.png        → Figure for paper
  {RESULTS_DIR}/f1_scores_bar_chart.png    → Figure for paper
  {RESULTS_DIR}/classification_results.json → All numbers

✅ WEEK 1 COMPLETE! You now have real experimental results for your paper.
👉 Next: Come back for the Week 2 notebook (Prompt Pipeline + API Integration)
''')